In [ ]:
"""
=============================================================================
COMPLETE U²-NET PAPER DETECTION PIPELINE FOR CHILDREN'S DRAWINGS
=============================================================================

INSTALLATION INSTRUCTIONS:
--------------------------
Run these commands in your terminal:

    pip install torch torchvision
    pip install opencv-python
    pip install matplotlib
    pip install numpy
    pip install pillow

WHAT THIS DOES:
---------------
1. Uses U²-Net (pre-trained salient object detection) to find paper
2. Segments paper from background (wood, table, etc.)
3. Normalizes orientation (portrait → landscape)
4. Cleans white background
5. Enhances crayon colors subtly
6. Resizes to match your dataset format

WHY U²-NET WORKS:
-----------------
- Trained to detect "main object" in image (paper = main object)
- Ignores background (wood, table, hands, etc.)
- Works even with colorful drawings on paper
- Better than Canny because it ignores drawing edges

=============================================================================
"""

import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
from torchvision import transforms
from PIL import Image


# =============================================================================
# STEP 1: LOAD U²-NET MODEL
# =============================================================================

class U2NET_PaperDetector:
    """
    U²-Net: Salient Object Detection Model
    - Pre-trained on 10,000+ images
    - Detects "main object" (paper) vs background
    """

    def __init__(self, use_hub=True):
        """
        Initialize U²-Net model

        Args:
            use_hub: If True, use torch.hub (default). If False, load manually.
        """
        print("🔄 Loading U²-Net model...")
        print("   (First time will download ~176MB)")

        if use_hub:
            # METHOD 1: Try torch hub first
            try:
                self.model = torch.hub.load('xuebinqin/U-2-Net', 'u2net', pretrained=True)
                print("✓ Loaded from torch hub")
            except Exception as e:
                print(f"⚠️  Torch hub failed: {e}")
                print("   Trying alternative loading method...")
                self._load_model_directly()
        else:
            # METHOD 2: Load directly without hub
            self._load_model_directly()

    def _load_model_directly(self):
        """
        Alternative loading method if torch hub fails
        Downloads model weights directly
        """
        import urllib.request
        from pathlib import Path

        # Create cache directory
        cache_dir = Path.home() / ".cache" / "u2net_models"
        cache_dir.mkdir(parents=True, exist_ok=True)

        model_path = cache_dir / "u2net.pth"

        # Download if not exists
        if not model_path.exists():
            print("📥 Downloading U²-Net weights directly...")
            url = "https://github.com/xuebinqin/U-2-Net/releases/download/v1.0/u2net.pth"
            urllib.request.urlretrieve(url, model_path)
            print(f"✓ Downloaded to {model_path}")

        # Import U²-Net architecture
        self.model = self._build_u2net_architecture()

        # Load weights
        print("📂 Loading model weights...")
        state_dict = torch.load(model_path, map_location='cpu')
        self.model.load_state_dict(state_dict)
        print("✓ Model loaded successfully")

    def _build_u2net_architecture(self):
        """
        Build U²-Net architecture manually
        This is a simplified version - full architecture is complex
        """
        # For now, fallback to torch hub with force_reload
        print("⚠️  Falling back to torch hub with force reload...")
        return torch.hub.load('xuebinqin/U-2-Net', 'u2net', pretrained=True, force_reload=True)

        # Set to evaluation mode (no training)
        self.model.eval()

        # Check if GPU is available
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model.to(self.device)

        print(f"✓ Model loaded on: {self.device}")

        # Define image preprocessing pipeline
        # U²-Net expects 320x320 images with normalized RGB values
        self.transform = transforms.Compose([
            transforms.Resize((320, 320)),  # Resize to model input size
            transforms.ToTensor(),  # Convert PIL to tensor [0, 1]
            transforms.Normalize(  # Normalize with ImageNet stats
                mean=[0.485, 0.456, 0.406],  # R, G, B means
                std=[0.229, 0.224, 0.225]  # R, G, B standard deviations
            )
        ])

    def predict_mask(self, img):
        """
        Get segmentation mask from U²-Net

        Args:
            img: RGB numpy array (H, W, 3)

        Returns:
            mask: Binary mask (0=background, 255=paper)
        """
        # Convert numpy array to PIL Image
        pil_img = Image.fromarray(img)
        original_size = pil_img.size  # (width, height)

        # Preprocess image
        input_tensor = self.transform(pil_img).unsqueeze(0)  # Add batch dimension
        input_tensor = input_tensor.to(self.device)

        # Run inference (no gradient computation needed)
        with torch.no_grad():
            # U²-Net outputs 7 saliency maps at different scales
            # d1 = main output, d2-d7 = auxiliary outputs for training
            d1, d2, d3, d4, d5, d6, d7 = self.model(input_tensor)

        # Use only the main output (d1)
        # Shape: [batch, 1, 320, 320]
        pred = d1[:, 0, :, :]

        # Apply sigmoid to get probability values [0, 1]
        pred = torch.sigmoid(pred)

        # Convert to numpy
        mask = pred.squeeze().cpu().numpy()

        # Resize mask back to original image size
        mask = cv2.resize(mask, original_size, interpolation=cv2.INTER_LINEAR)

        # Convert to 0-255 range
        mask = (mask * 255).astype(np.uint8)

        # Apply binary threshold: > 127 = paper (255), else background (0)
        _, mask_binary = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)

        return mask_binary


# =============================================================================
# STEP 2: SEGMENT PAPER USING U²-NET MASK
# =============================================================================

def segment_paper_from_mask(img, mask, debug=False):
    """
    Extract paper region using the predicted mask

    Args:
        img: Original RGB image
        mask: Binary mask from U²-Net
        debug: Show visualization steps

    Returns:
        Cropped paper image with white background
    """

    # --- Find the paper contour ---
    # findContours detects boundaries in binary mask
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        raise RuntimeError("❌ No contours found in mask - paper not detected")

    # Get the largest contour (should be the paper)
    largest_contour = max(contours, key=cv2.contourArea)

    # Get bounding rectangle of the contour
    x, y, w, h = cv2.boundingRect(largest_contour)

    # Add small padding to avoid cutting edges
    pad = 10
    x = max(0, x - pad)
    y = max(0, y - pad)
    w = min(img.shape[1] - x, w + 2 * pad)
    h = min(img.shape[0] - y, h + 2 * pad)

    # Crop both image and mask
    cropped_img = img[y:y + h, x:x + w].copy()
    cropped_mask = mask[y:y + h, x:x + w]

    # --- Apply mask to remove background ---
    # Create white background
    white_bg = np.full_like(cropped_img, 255, dtype=np.uint8)

    # Where mask=255 use image, where mask=0 use white
    # cropped_mask[..., None] expands shape from (H,W) to (H,W,1) for broadcasting
    result = np.where(cropped_mask[..., None] == 255, cropped_img, white_bg)

    if debug:
        plt.figure(figsize=(15, 5))

        plt.subplot(131)
        plt.imshow(img)
        plt.title("Original Image")
        plt.axis('off')

        plt.subplot(132)
        plt.imshow(mask, cmap='gray')
        plt.title("U²-Net Mask (White=Paper)")
        plt.axis('off')

        plt.subplot(133)
        plt.imshow(result)
        plt.title("Segmented Paper")
        plt.axis('off')

        plt.tight_layout()
        plt.show()

    return result


# =============================================================================
# STEP 3: NORMALIZE ORIENTATION (Portrait → Landscape)
# =============================================================================

def normalize_orientation(img):
    """
    Rotate paper to landscape if it's portrait

    Most papers are landscape, so if height > width, rotate 90° clockwise
    """
    h, w = img.shape[:2]

    if h > w:
        # Portrait detected, rotate to landscape
        img = cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)
        print(f"   ↻ Rotated from portrait ({w}×{h}) to landscape ({h}×{w})")
    else:
        print(f"   → Already landscape ({w}×{h})")

    return img


# =============================================================================
# STEP 4: CLEAN WHITE BACKGROUND
# =============================================================================

def clean_white_background(img):
    """
    Make white/light areas pure white (255, 255, 255)
    Preserves colored crayon strokes

    Uses LAB color space:
    - L = Lightness (0=black, 100=white)
    - A = Green to Red
    - B = Blue to Yellow
    """

    # Convert RGB → LAB color space
    # LAB is better for separating brightness from color
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l_channel, a_channel, b_channel = cv2.split(lab)

    # Find pixels that are already bright (likely paper background)
    # L > 200 means very bright (paper)
    bright_mask = l_channel > 200

    # Set those pixels to maximum brightness (255 = pure white)
    l_channel[bright_mask] = 255

    # Merge channels back
    lab_cleaned = cv2.merge((l_channel, a_channel, b_channel))

    # Convert LAB → RGB
    result = cv2.cvtColor(lab_cleaned, cv2.COLOR_LAB2RGB)

    return result


# =============================================================================
# STEP 5: ENHANCE CRAYON COLORS (SUBTLE)
# =============================================================================

def enhance_crayon_colors(img):
    """
    Gently boost saturation to make crayons more vibrant

    Uses HSV color space:
    - H = Hue (color type: red, blue, etc.)
    - S = Saturation (color intensity)
    - V = Value (brightness)
    """

    # Convert RGB → HSV
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    h_channel, s_channel, v_channel = cv2.split(hsv)

    # --- Boost Saturation (makes colors more vivid) ---
    # Multiply by 1.15 = 15% more saturated
    # Add +5 to lift low saturation areas slightly
    s_channel = s_channel.astype(np.float32)
    s_channel = s_channel * 1.15 + 5
    s_channel = np.clip(s_channel, 0, 255)  # Keep in valid range [0, 255]
    s_channel = s_channel.astype(np.uint8)

    # --- Slight Brightness Adjustment ---
    # Multiply by 1.02 = 2% brighter
    v_channel = v_channel.astype(np.float32)
    v_channel = v_channel * 1.02
    v_channel = np.clip(v_channel, 0, 255)
    v_channel = v_channel.astype(np.uint8)

    # Merge channels back
    hsv_enhanced = cv2.merge((h_channel, s_channel, v_channel))

    # Convert HSV → RGB
    result = cv2.cvtColor(hsv_enhanced, cv2.COLOR_HSV2RGB)

    return result


# =============================================================================
# STEP 6: RESIZE TO MATCH YOUR DATASET FORMAT
# =============================================================================

def resize_to_dataset(img, target_width=512):
    """
    Resize image to target width while maintaining aspect ratio

    Args:
        img: Input image
        target_width: Desired width (default 512px)

    Returns:
        Resized image
    """
    h, w = img.shape[:2]

    # Calculate scale factor
    scale = target_width / w

    # Calculate new dimensions
    new_width = target_width
    new_height = int(h * scale)

    # Resize using high-quality interpolation
    # INTER_AREA is best for downscaling (reduces aliasing)
    resized = cv2.resize(img, (new_width, new_height), interpolation=cv2.INTER_AREA)

    print(f"   📐 Resized: {w}×{h} → {new_width}×{new_height}")

    return resized


# =============================================================================
# STEP 7: COMPLETE PIPELINE
# =============================================================================

def full_pipeline(image_path, detector, debug=False, save_path="output_final.png"):
    """
    Complete processing pipeline

    Args:
        image_path: Path to input image
        detector: U2NET_PaperDetector instance
        debug: Show intermediate steps
        save_path: Where to save final output

    Returns:
        Final processed image
    """

    print(f"\n{'=' * 70}")
    print(f"🎨 Processing: {image_path}")
    print(f"{'=' * 70}")

    # --- LOAD IMAGE ---
    img = cv2.imread(image_path)
    if img is None:
        raise RuntimeError(f"❌ Cannot read image: {image_path}")

    # Convert BGR (OpenCV default) → RGB (standard)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    print(f"✓ Loaded image: {img.shape[1]}×{img.shape[0]} pixels")

    # --- STEP 1: Detect Paper with U²-Net ---
    print("\n🔍 Step 1: Detecting paper with U²-Net...")
    mask = detector.predict_mask(img)
    print(f"✓ Mask generated: {np.sum(mask > 0)} pixels detected as paper")

    # --- STEP 2: Segment Paper ---
    print("\n✂️  Step 2: Segmenting paper from background...")
    paper = segment_paper_from_mask(img, mask, debug=debug)
    print(f"✓ Paper extracted: {paper.shape[1]}×{paper.shape[0]}")

    # --- STEP 3: Normalize Orientation ---
    print("\n🔄 Step 3: Normalizing orientation...")
    paper = normalize_orientation(paper)

    # --- STEP 4: Clean Background ---
    print("\n🧹 Step 4: Cleaning white background...")
    paper = clean_white_background(paper)
    print("✓ Background cleaned")

    # --- STEP 5: Enhance Colors ---
    print("\n🎨 Step 5: Enhancing crayon colors...")
    paper = enhance_crayon_colors(paper)
    print("✓ Colors enhanced")

    # --- STEP 6: Resize ---
    print("\n📏 Step 6: Resizing to dataset format...")
    final = resize_to_dataset(paper, target_width=512)

    # --- SAVE OUTPUT ---
    print(f"\n💾 Saving output to: {save_path}")
    output_bgr = cv2.cvtColor(final, cv2.COLOR_RGB2BGR)
    cv2.imwrite(save_path, output_bgr)

    print(f"\n{'=' * 70}")
    print("✅ PIPELINE COMPLETE!")
    print(f"{'=' * 70}\n")

    return final


# =============================================================================
# STEP 8: VISUALIZATION FUNCTION
# =============================================================================

def visualize_all_steps(image_path, detector):
    """
    Show all pipeline steps side-by-side for debugging
    """
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Run each step
    mask = detector.predict_mask(img)
    paper = segment_paper_from_mask(img, mask, debug=False)
    paper_oriented = normalize_orientation(paper)
    paper_clean = clean_white_background(paper_oriented)
    paper_enhanced = enhance_crayon_colors(paper_clean)
    final = resize_to_dataset(paper_enhanced)

    # Create visualization grid
    plt.figure(figsize=(18, 10))

    plt.subplot(2, 4, 1)
    plt.imshow(img)
    plt.title("1. Original Input", fontsize=12, fontweight='bold')
    plt.axis('off')

    plt.subplot(2, 4, 2)
    plt.imshow(mask, cmap='gray')
    plt.title("2. U²-Net Mask", fontsize=12, fontweight='bold')
    plt.axis('off')

    plt.subplot(2, 4, 3)
    plt.imshow(paper)
    plt.title("3. Segmented Paper", fontsize=12, fontweight='bold')
    plt.axis('off')

    plt.subplot(2, 4, 4)
    plt.imshow(paper_oriented)
    plt.title("4. Oriented", fontsize=12, fontweight='bold')
    plt.axis('off')

    plt.subplot(2, 4, 5)
    plt.imshow(paper_clean)
    plt.title("5. Background Cleaned", fontsize=12, fontweight='bold')
    plt.axis('off')

    plt.subplot(2, 4, 6)
    plt.imshow(paper_enhanced)
    plt.title("6. Colors Enhanced", fontsize=12, fontweight='bold')
    plt.axis('off')

    plt.subplot(2, 4, 7)
    plt.imshow(final)
    plt.title("7. Final Resized", fontsize=12, fontweight='bold')
    plt.axis('off')

    # Show target reference
    plt.subplot(2, 4, 8)
    plt.text(0.5, 0.5, f"Target:\n512px wide\n~362px height\n\nActual:\n{final.shape[1]}×{final.shape[0]}",
             ha='center', va='center', fontsize=14, fontweight='bold')
    plt.title("8. Dimensions", fontsize=12, fontweight='bold')
    plt.axis('off')

    plt.tight_layout()
    plt.savefig('pipeline_steps_visualization.png', dpi=150, bbox_inches='tight')
    plt.show()

    return final


# =============================================================================
# MAIN EXECUTION
# =============================================================================

if __name__ == "__main__":

    # --- INITIALIZE DETECTOR (Do this ONCE) ---
    print("🚀 Initializing U²-Net Paper Detector...")
    detector = U2NET_PaperDetector()

    # --- PROCESS SINGLE IMAGE ---
    try:
        final_result = full_pipeline(
            image_path="wood_bg1.jpeg",  # ← Change this to your image
            detector=detector,
            debug=True,  # Set False to skip intermediate plots
            save_path="final_output.png"
        )

        # Display final result
        plt.figure(figsize=(10, 7))
        plt.imshow(final_result)
        plt.title("Final Result", fontsize=16, fontweight='bold')
        plt.axis('off')
        plt.tight_layout()
        plt.show()

        # Show all steps
        print("\n📊 Generating complete visualization...")
        visualize_all_steps("wood_bg1.jpeg", detector)

    except Exception as e:
        print(f"\n❌ Error: {e}")
        import traceback

        traceback.print_exc()


# =============================================================================
# BATCH PROCESSING (Multiple Images)
# =============================================================================

def process_batch(image_folder, output_folder, detector):
    """
    Process all images in a folder

    Args:
        image_folder: Folder containing input images
        output_folder: Where to save outputs
        detector: U2NET_PaperDetector instance
    """
    import os
    from pathlib import Path

    # Create output folder if doesn't exist
    Path(output_folder).mkdir(parents=True, exist_ok=True)

    # Get all image files
    extensions = ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']
    image_files = []
    for ext in extensions:
        image_files.extend(Path(image_folder).glob(f'*{ext}'))

    print(f"\n📁 Found {len(image_files)} images to process")

    # Process each image
    success_count = 0
    fail_count = 0

    for i, img_path in enumerate(image_files, 1):
        try:
            print(f"\n[{i}/{len(image_files)}] Processing: {img_path.name}")

            output_path = Path(output_folder) / f"processed_{img_path.name}"

            final = full_pipeline(
                image_path=str(img_path),
                detector=detector,
                debug=False,
                save_path=str(output_path)
            )

            success_count += 1

        except Exception as e:
            print(f"❌ Failed: {e}")
            fail_count += 1
            continue

    print(f"\n{'=' * 70}")
    print(f"✅ Successfully processed: {success_count}")
    print(f"❌ Failed: {fail_count}")
    print(f"{'=' * 70}")

# Example usage:
# detector = U2NET_PaperDetector()
# process_batch("input_images/", "output_images/", detector)